REdes neuronales para predicción de ligas

Para crear un programa en Python que realice la predicción de ligas (relaciones entre los artículos científicos), usaremos redes neuronales y procesamiento de texto para extraer las características relevantes de los artículos y sus relaciones. Dado que los datos tienen diferentes campos (como el título, el área, las palabras clave, etc.), una red neuronal puede aprender patrones complejos para predecir qué artículos están más relacionados entre sí.

En este caso, te guiaré en la creación de un modelo de red neuronal en Python, usando bibliotecas como TensorFlow para las redes neuronales, sklearn para el preprocesamiento de datos, y pandas para manejar los datos. Asumiré que las predicciones de las "ligas" se basan en relaciones entre artículos, como las coocurrencias de autores, áreas, o palabras clave.

Paso 1: Preprocesar los datos
Primero, es necesario preprocesar los datos para crear características que la red neuronal pueda utilizar. Podríamos usar las palabras clave, los títulos y las áreas de los artículos para construir representaciones vectoriales que la red neuronal pueda procesar.

In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import accuracy_score
import tensorflow as tf

# Cargar el dataset
df = pd.read_csv('C:/Users/pilarang/0-ProyAcademicosGrafo-Integrado/3-AplicacionWebDash/datasets/UNAM_Completo_Corregido-v2-utf8.csv')

# Preprocesar los datos
# Usamos las columnas 'Titulo', 'Area', y 'Keywords' para obtener características
df['Keywords'] = df['Keywords'].fillna('')  # Rellenar valores faltantes
df['Texto'] = df['Titulo'] + " " + df['Keywords']  # Concatenar título y keywords para el modelo

# Vectorizar las columnas de texto (usamos TF-IDF para convertir a vectores)
tfidf = TfidfVectorizer(max_features=1000)  # Limitar las características a 1000
X = tfidf.fit_transform(df['Texto']).toarray()

# Si las ligas están indicadas en alguna columna, como 'Area' o 'Subarea', convertimos en categorías
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['Area'])  # Asumimos que 'Area' es nuestra etiqueta a predecir

# Dividir los datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


Paso 2: Construcción del modelo de red neuronal
A continuación, definimos la red neuronal. Usaremos una red neuronal densa con varias capas.

In [8]:
# Definir el modelo de red neuronal
model = Sequential()

# Capa de entrada
model.add(Dense(512, input_dim=X_train.shape[1], activation='relu'))

# Capa oculta
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.5))  # Dropout para evitar sobreajuste

# Capa de salida
model.add(Dense(len(label_encoder.classes_), activation='softmax'))  # 'softmax' para clasificación multiclase

# Compilar el modelo
model.compile(optimizer=Adam(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Entrenar el modelo
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))


C:\Users\pilarang\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.2815 - loss: 1.8121 - val_accuracy: 0.5975 - val_loss: 1.1459
Epoch 2/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7062 - loss: 0.8892 - val_accuracy: 0.6879 - val_loss: 0.8994
Epoch 3/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8239 - loss: 0.5384 - val_accuracy: 0.7231 - val_loss: 0.8354
Epoch 4/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9023 - loss: 0.3184 - val_accuracy: 0.7469 - val_loss: 0.8352
Epoch 5/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9378 - loss: 0.2078 - val_accuracy: 0.7555 - val_loss: 0.8867
Epoch 6/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9464 - loss: 0.1648 - val_accuracy: 0.7488 - val_loss: 0.9401
Epoch 7/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9591 - loss: 0.1288 - val_accuracy: 0.7517 - val_loss: 0.9916
Epoch 8/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9632 - loss: 0.1197 - val_accuracy: 0.

Paso 3: Evaluar el modelo
Después de entrenar el modelo, evaluamos su desempeño en el conjunto de prueba.

python
Copiar


In [11]:
# Evaluar el modelo
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

# Calcular la precisión
accuracy = accuracy_score(y_test, y_pred_classes)
print(f'Accuracy: {accuracy:.2f}')


33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
Accuracy: 0.76


Paso 4: Predicciones
Para hacer predicciones con el modelo, simplemente usamos el método predict para un nuevo artículo.

In [14]:
def predecir_area(titulo, keywords):
    texto = titulo + " " + keywords
    texto_vec = tfidf.transform([texto]).toarray()
    prediccion = model.predict(texto_vec)
    area_predicha = label_encoder.inverse_transform([np.argmax(prediccion)])
    return area_predicha[0]

# Ejemplo de predicción
titulo = "Estudio sobre Redes Neuronales en IA"
keywords = "redes neuronales, inteligencia artificial"
area_predicha = predecir_area(titulo, keywords)
print(f'El área predicha para el artículo es: {area_predicha}')


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
El área predicha para el artículo es: ISBD


Paso 5: Visualización del grafo
Si deseas realizar alguna visualización de las relaciones entre los artículos, puedes usar el código que mencionaste al principio para crear y visualizar el grafo de artículos y autores.

In [17]:
import networkx as nx
import dash_cytoscape as cyto

# Suponiendo que tienes un grafo `G` con los artículos y sus relaciones
# Convertir el grafo de NetworkX a formato compatible con Cytoscape para visualización en Dash
cy = convertGraphToCY(G)

# Visualización de relaciones en un componente Dash Cytoscape
app = Dash(__name__)

app.layout = html.Div([
    cyto.Cytoscape(
        id='cytoscape',
        elements=cy['elements'],
        layout={'name': 'breadthfirst'},
        style={'width': '100%', 'height': '600px'}
    )
])

if __name__ == '__main__':
    app.run_server(debug=True)


NameError: name 'convertGraphToCY' is not defined

Resumen:
Hemos preprocesado el conjunto de datos de artículos para convertir el texto en vectores utilizando TF-IDF.
Creamos un modelo de red neuronal con Keras para predecir las áreas de los artículos (o las ligas entre ellos, dependiendo de cómo definas el objetivo).
Evaluamos el modelo y mostramos cómo realizar predicciones.
También hemos mostrado cómo visualizar el grafo con Dash y Cytoscape.